# Tutorial 1: Keys, Values, and the Keyspace (Lecture 12)

Every tutorial up to this point has gone through MySQL. We described a schema up front, created tables, declared types and keys, and only then put data in.

Redis asks for none of that. There is no schema, no table, and no `CREATE` statement. There is a key, and there is a value stored against it, and the first write is also the moment the key comes into existence.

This notebook covers the basics: connecting, writing and reading a value, naming keys sensibly, and looking around the keyspace. Tutorial 2 picks up from here with the data structures that make Redis more than a dictionary.

**Before you start**, bring the stack up from the `redis-dev` folder:

```bash
docker compose up -d --force-recreate
```

## Connecting

We use the `redis` package for Python, usually called redis-py. It plays the same role `pymysql` did on the MySQL side.

Notice how much less there is to set up. No cursor, no separate connection object to keep track of, and no database name. In Redis the client object *is* the interface, and `r` below is the only handle we will need.

In [1]:
import redis

REDIS_HOST = "redis"      # use "localhost" if running outside the jupyter container
REDIS_PORT = 6379

r = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    decode_responses=True,
)

r.ping()

True

`ping()` returning `True` means the server answered. If it raises a `ConnectionError` instead, the `redis` service is not up, so check `docker compose ps`.

### Why `decode_responses=True`

This setting matters more than it looks. Redis stores and returns raw bytes. Without this flag every value you read comes back as a bytes object, so a stored name reads as `b'Felton'` rather than `'Felton'`, and comparisons like `value == "Felton"` quietly fail.

Setting it once at connection time means every result from here on is an ordinary Python string. The cell below shows the difference by opening a second client without it.

In [2]:
r.set("demo:name", "Felton")

r_bytes = redis.Redis(host=REDIS_HOST, port=REDIS_PORT)   # no decode_responses

print("with decode_responses=True :", repr(r.get("demo:name")))
print("without it                 :", repr(r_bytes.get("demo:name")))

print()
print('r.get(...) == "Felton"       :', r.get("demo:name") == "Felton")
print('r_bytes.get(...) == "Felton" :', r_bytes.get("demo:name") == "Felton")

with decode_responses=True : 'Felton'
without it                 : b'Felton'

r.get(...) == "Felton"       : True
r_bytes.get(...) == "Felton" : False


That second `False` is the bug this flag saves you from. It is not an error, nothing is raised, the comparison simply never matches.

## Keys and Values

The whole model is a key and a value stored against it. `set` writes, `get` reads.

In [3]:
r.set("customer:C25:city", "HERSHEY")
r.get("customer:C25:city")

'HERSHEY'

Nothing was created before that write. There is no table, no schema, and no column list. The key did not exist, and after `set` it does.

`exists` asks whether a key is there, and `delete` removes it. Both answer with a count rather than `True` or `False`, which is worth noticing now so the `1` and `0` below do not look odd later.

In [4]:
print("exists before delete:", r.exists("customer:C25:city"))
print("delete returned     :", r.delete("customer:C25:city"))
print("exists after delete :", r.exists("customer:C25:city"))

exists before delete: 1
delete returned     : 1
exists after delete : 0


Reading a key that is not there is not an error either. You get `None` back, the same as a missing key in a Python dictionary accessed with `.get()`.

In [5]:
print(r.get("customer:C25:city"))
print(r.get("a:key:that:was:never:written"))

None
None


## Naming Keys

The colons in `customer:C25:city` are a convention, not a rule. Redis sees one flat string and attaches no meaning to the colons at all.

The convention still earns its place. A real keyspace holds a great many keys with no schema to organise them, and a naming scheme is the only structure it will ever have. The usual shape is `object:id:field`.

Below we load the real customer data from the orders dataset, the same CSV used all through the MySQL tutorials, and write one key per field.

In [6]:
import csv

csv_path =  "/home/jovyan/iem4723/mysql-dev/data/orders/Customer.csv" #"../../data/orders/Customer.csv"

with open(csv_path, newline="") as f:
    customer_rows = list(csv.DictReader(f))

for row in customer_rows:
    num = row["Customer_Num"]
    r.set(f"customer:{num}:first_name", row["Customer_FirstName"])
    r.set(f"customer:{num}:last_name", row["Customer_LastName"])
    r.set(f"customer:{num}:city", row["Customer_City"])
    r.set(f"customer:{num}:state", row["Customer_State"])

print(f"wrote 4 keys each for {len(customer_rows)} customers")

wrote 4 keys each for 10 customers


In [7]:
print(r.get("customer:C25:first_name"), r.get("customer:C25:last_name"))
print(r.get("customer:C25:city"), r.get("customer:C25:state"))

Bob Felton
HERSHEY PA


Compare that to the MySQL version of the same lookup. There we wrote a `SELECT` with a `WHERE` clause and the server searched for the row. Here the key *is* the address of the value, so there is nothing to search.

That is the trade. Looking up a customer by number is immediate. Asking *which customers live in Pennsylvania* has no answer at all with keys shaped like this, because nothing indexes the value side. In SQL that is just another `WHERE` clause.

## Working With the Keyspace

`keys` takes a pattern and returns every key matching it.

In [8]:
sorted(r.keys("customer:C25:*"))

['customer:C25:city',
 'customer:C25:first_name',
 'customer:C25:last_name',
 'customer:C25:state']

In [9]:
print("customers seen:", len(r.keys("customer:*:city")))
print("keys in this database:", r.dbsize())

customers seen: 10
keys in this database: 41


**Treat `keys()` as a debugging tool only.** It walks the entire keyspace to answer, so on a production server holding millions of keys it is genuinely expensive, and because Redis handles one command at a time it blocks everything else while it runs. The production alternative is `scan_iter()`, which walks the keyspace in small batches instead.

In [10]:
# the same question, asked the way you would ask it on a real server
for key in sorted(r.scan_iter(match="customer:C2501:*")):
    print(key, "->", r.get(key))

customer:C2501:city -> PITTSBURGH
customer:C2501:first_name -> John
customer:C2501:last_name -> Grant
customer:C2501:state -> PA


`type` reports what kind of value is stored against a key. Everything we have written so far is a string; Tutorial 2 is about the other four.

In [11]:
print(r.type("customer:C25:city"))
print(r.type("no:such:key"))     # 'none' when the key does not exist

string
none


## Clearing Up

`flushdb` empties the current database. It does not ask for confirmation and there is no undo, in much the same spirit as a `DELETE` with no `WHERE` clause.

Note that it empties *the entire database the client is connected to*. 

In [12]:
print("dbsize :", r.dbsize())

r.flushdb()

print("dbsize :", r.dbsize())

dbsize : 41
dbsize : 0


## Remember these important pointers

- There is no schema like MySQL, everything is in keys. The first write makes the key.
- Default responses is in bytes and not in characters.
- `decode_responses=True` at connection time saves you from comparing strings against bytes.
- Key naming is the only organisation a keyspace has, so `object:id:field` naming convention is worth adhering to.
- Lookup by key is immediate; searching by value is not something this model does.
- `keys()` for learning, `scan_iter()` for real servers, and `flushdb()` with care.